# 20 — WAL & Recovery

### Why this exists
A write-ahead log makes recovery possible by recording intent before the data page is considered durable. This lab simulates commit boundaries and crash recovery.

### Learning objectives
model log records; replay committed work; distinguish durable from merely acknowledged work

### Prerequisite
Python lists and state reconstruction

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/20-wal-recovery.svg)

> **Question to carry through the lab:** After a crash, which operations should exist and which should disappear?

## Capability contract

**Capability:** C05 — Storage Consistency & Distributed Systems  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Explain a distributed trade-off


## Mental model
The essential invariant is ordering: the durable log must contain enough information to reconstruct committed state before the corresponding data mutation is treated as durable.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S08 — Replication and consistency (DATA-1110)** → [`../work_simulations/08_acme_commerce_replica_lag/README.md`](../work_simulations/08_acme_commerce_replica_lag/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
def append(log,tx,op,key,value): log.append({"tx":tx,"op":op,"key":key,"value":value})
log=[]
append(log,"t1","SET","balance",100)
append(log,"t1","COMMIT","",None)
append(log,"t2","SET","balance",20)
print(log)

In [ ]:
def recover(log):
    committed={r["tx"] for r in log if r["op"]=="COMMIT"}
    state={}
    for r in log:
        if r["tx"] in committed and r["op"]=="SET": state[r["key"]]=r["value"]
    return state
print("recovered",recover(log))

In [ ]:
# Crash before t3 commit: its mutation is present in the log but must not become visible.
append(log,"t3","SET","balance",999)
print("after crash",recover(log))

In [ ]:
# Crash after commit: replay must make the operation visible.
append(log,"t3","COMMIT","",None)
print("after committed recovery",recover(log))

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## Your task

**Goal:** Make crash recovery detect incomplete log records.

**Do this:** Add checksums or sequence numbers, simulate an incomplete tail, and define what recovery should do with it.

**Before you finish, save:** your prediction, the key measurement, the failure/edge case you tested, your diagnosis, and the production implication.

**Done when:** another engineer can reproduce your result and understand why you made the decision.

### Exercise detail

## 8. Production bridge
Production WAL systems add fsync/durability semantics, checksums, segment management, checkpoints, replay limits and operational recovery tooling.

## 9. Independent challenge
Add checksums or record sequence numbers to detect a torn final log record. Decide what the system should do with an incomplete tail after a crash.

### Evidence to keep
recovery state before/after commit, incomplete transaction handling, and the durability boundary you would expose to callers

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.

---

**Check your work:** [Open the reference solution](https://github.com/ankit-rathi/data-systems-lab/blob/main/solutions/notebooks/20.md) — only after you have attempted the exercise.